# Testing Some of the Most Famous Running Myths
## Social Data Analysis — DTU 2026
**Team:** Marta Arana · Esben · Sergi  
**Race:** Lyngby Half Marathon 2026 — April 26 · 21.1 km

---


## 1. Motivation

Running culture is full of advice passed down through generations of athletes: *"hill training will save you on race day"*, *"experienced runners never blow up"*, *"more kilometres always means a better finish"*. But how much of this is myth — and how much is backed by data?

On April 26, 2026, a group of 25 runners — friends, colleagues, and rivals — lined up at the Lyngby Half Marathon in Denmark. They ranged from sub-80-minute elites to first-timers chasing a finish line, and every one of them had been quietly tracking their training for weeks beforehand. We collected their GPS data, training logs, and race splits to put four popular running myths to the test.

**Research questions:**
1. Do experienced runners pace themselves better than newcomers?
2. Does hill training actually help on a hilly course?
3. Is higher training volume always associated with faster finish times?
4. Do interval trainers show smarter race-day split strategies?

This notebook documents the full data pipeline — from raw GPX files and survey responses to interactive visualisations — and presents our findings with both statistical rigour and narrative context.


## 2. Basic Statistics

### 2.1 Dataset Construction

The dataset combines two sources:
- **Survey (Google Forms):** training habits, experience level, weekly mileage, target finish time, hill/interval training flags
- **GPS (Garmin .gpx files):** per-kilometre splits, heart rate, cadence, elevation

Each runner's `.gpx` file is parsed using Python's built-in `xml.etree.ElementTree`. Haversine distance is computed between consecutive trackpoints to derive kilometre markers.


In [ ]:
import pandas as pd
import numpy as np
import pickle, os

DATA_DIR = '/sessions/vigilant-awesome-mayer/mnt/data/processed'
df = pd.read_csv(f'{DATA_DIR}/lyngby_runners_2026.csv')

print(f"Runners: {len(df)}")
print(f"Columns: {df.shape[1]}")
print(f"GPS data available: {df['has_gps'].sum()} runners")
print()
print(df[['name','archetype','finish_min','km_per_week','training_weeks','trained_hills','trains_intervals']].to_string(index=False))


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

mpl.rcParams.update({
    'figure.facecolor': '#111111',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333333',
    'axes.labelcolor':  '#cccccc',
    'axes.titlecolor':  '#ffffff',
    'xtick.color':      '#888888',
    'ytick.color':      '#888888',
    'text.color':       '#cccccc',
    'grid.color':       '#2a2a2a',
    'grid.linestyle':   '--',
    'grid.linewidth':   0.6,
    'legend.facecolor': '#222222',
    'legend.edgecolor': '#444444',
    'legend.labelcolor':'#cccccc',
    'font.family':      'sans-serif',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

NIKE  = '#FA5400'
GRI   = '#00C8FF'
BEL   = '#888888'
GRAD  = '#FF6B35'
WHITE = '#ffffff'

ARCH_COLORS = {
    'The Experienced': GRI,
    'The Grinders':    NIKE,
    'The Believers':   '#aaaaaa',
}
arch_order = ['The Experienced', 'The Grinders', 'The Believers']
print("Theme loaded — Nike dark mode active")


In [ ]:
# Distribution overview
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Runner Distribution Overview', fontsize=14, color=WHITE, fontweight='bold', y=1.02)

# Finish times histogram
axes[0].hist(df['finish_min'], bins=10, color=NIKE, alpha=0.85, edgecolor='#333333', linewidth=0.7)
axes[0].axvline(df['finish_min'].mean(), color=GRI, linewidth=2, linestyle='--',
                label=f'Mean {df["finish_min"].mean():.0f} min')
axes[0].set_title('Finish Time Distribution')
axes[0].set_xlabel('Finish Time (min)')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].grid(True, alpha=0.4)

# Gender split
gender_counts = df['gender'].value_counts()
colors_g = [NIKE if g=='M' else GRI for g in gender_counts.index]
axes[1].bar(gender_counts.index, gender_counts.values, color=colors_g, alpha=0.85, edgecolor='#333333')
axes[1].set_title('Gender Split')
axes[1].set_ylabel('Count')
axes[1].grid(True, alpha=0.4, axis='y')

# Archetype counts
arch_counts = df['archetype'].value_counts()
c_arch = [ARCH_COLORS.get(a, BEL) for a in arch_counts.index]
axes[2].bar(arch_counts.index, arch_counts.values, color=c_arch, alpha=0.85, edgecolor='#333333')
axes[2].set_title('Runner Archetypes')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', labelsize=8)
axes[2].grid(True, alpha=0.4, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
# Archetype summary statistics
print("Summary statistics by archetype:")
print(df.groupby('archetype')[['finish_min','km_per_week','training_weeks','split_ratio']].mean().round(2).to_string())
print()
print("Hill trained:", df['trained_hills'].sum(), "runners")
print("Interval trained:", df['trains_intervals'].sum(), "runners")


## 3. Data Analysis

### 3.1 GPX Parsing Pipeline

Each `.gpx` file is parsed to extract:
- **Coordinates** (lat/lon) every ~5 seconds
- **Heart rate** from the Garmin extension namespace `{http://www.garmin.com/xmlschemas/TrackPointExtension/v1}`
- **Cadence** from the same extension
- **Elevation** from the `<ele>` tag

Distance between consecutive trackpoints is computed using the Haversine formula.


In [ ]:
# Load pre-computed GPX data
with open(f'{DATA_DIR}/gpx_data.pkl', 'rb') as f:
    gpx = pickle.load(f)

names_with_gps = list(gpx.keys())
print(f"Runners with GPS data: {len(names_with_gps)}")

# Sample split data
sample = gpx['Marta Arana']
print(f"\nMarta Arana: {sample['n_points']} trackpoints, {len(sample['splits'])} km splits")
print("\nFirst 5 splits:")
print(f"{'km':>4} {'pace':>8} {'HR':>6} {'cad':>6} {'ele':>7}")
for s in sample['splits'][:5]:
    print(f"{s['km']:>4} {s['pace']:>8.2f} {s['hr']:>6.0f} {s['cad']:>6.0f} {s['ele']:>7.1f}")


In [ ]:
# Course elevation profile (Marta's route as course representative)
splits_m = gpx['Marta Arana']['splits']
kms_m    = [s['km']   for s in splits_m]
eles_m   = [s['ele']  for s in splits_m]
paces_m  = [s['pace'] for s in splits_m]

fig, ax = plt.subplots(figsize=(14, 4))
fig.patch.set_facecolor('#111111')
ax.set_facecolor('#1a1a1a')

mn, mx = min(paces_m), max(paces_m)

def pace_color(t):
    if t < 0.33:
        g = int(200 - t/0.33*80)
        return f'#{0x28:02x}{g:02x}{0x28:02x}'
    elif t < 0.66:
        tt = (t-0.33)/0.33
        r = int(0xFA*tt + 0xA0*(1-tt))
        g = int(0x80*(1-tt) + 0x80)
        return f'#{r:02x}{g:02x}{0x28:02x}'
    else:
        tt = (t-0.66)/0.34
        r = 0xFA
        g = int(0x54*(1-tt))
        return f'#{r:02x}{g:02x}{0x28:02x}'

norm_p = [(p-mn)/(mx-mn+0.001) for p in paces_m]
for i in range(len(kms_m)-1):
    col = pace_color(norm_p[i])
    ax.fill_between([kms_m[i], kms_m[i+1]], [eles_m[i], eles_m[i+1]], alpha=0.7, color=col)
    ax.plot([kms_m[i], kms_m[i+1]], [eles_m[i], eles_m[i+1]], color=col, linewidth=2)

ax.axvspan(13, 16, alpha=0.15, color=NIKE, label='Killer Climb (km 13-16)')
ax.set_xlabel('Distance (km)')
ax.set_ylabel('Elevation (m)')
ax.set_title('Course Elevation Profile — Coloured by Pace (green=fast, red=slow)', color=WHITE, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 3.2 Prediction Model

We build an OLS regression to predict finish times from training data. The model uses `km_per_week` and `training_weeks` as features. The final predicted time is a blend incorporating each runner's stated target time:

**predicted = 0.4 × target + 0.6 × model_prediction**

This acknowledges that runners often have good self-knowledge about their own fitness. The formula is not shown in the public website — only the predicted vs actual scatter.


In [ ]:
from numpy.linalg import lstsq

features = df[['km_per_week', 'training_weeks']].values
y        = df['finish_min'].values
X        = np.column_stack([np.ones(len(features)), features])
w, _, _, _ = lstsq(X, y, rcond=None)
model_pred = X @ w
predicted  = 0.4 * df['target_min'].values + 0.6 * model_pred

corr = np.corrcoef(predicted, y)[0, 1]
rmse = np.sqrt(np.mean((y - predicted)**2))
print(f"OLS weights: intercept={w[0]:.2f}, km_per_week={w[1]:.3f}, training_weeks={w[2]:.3f}")
print(f"Pearson r (predicted vs actual): {corr:.3f}")
print(f"RMSE: {rmse:.2f} min")

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#111111')
ax.set_facecolor('#1a1a1a')

for arch in arch_order:
    idx = df['archetype'] == arch
    ax.scatter(df[idx]['finish_min'], predicted[idx],
               c=ARCH_COLORS[arch], s=85, alpha=0.85,
               edgecolors='#333333', linewidth=0.5, label=arch, zorder=3)

lims = [min(y.min(), predicted.min())-2, max(y.max(), predicted.max())+2]
ax.plot(lims, lims, color='#666666', linewidth=1.5, linestyle='--', label='Perfect prediction')

# Annotate some runners
for _, row in df.iterrows():
    idx2 = df.index.get_loc(_)
    ax.annotate(row['name'].split()[0], (row['finish_min'], predicted[idx2]),
                fontsize=7, color='#888888', alpha=0.7,
                xytext=(3, 3), textcoords='offset points')

from matplotlib.lines import Line2D
legend_elements = [Line2D([0],[0], marker='o', color='w', markerfacecolor=ARCH_COLORS[a],
                          markersize=8, label=a) for a in arch_order]
legend_elements.append(Line2D([0],[0], color='#666666', linestyle='--', label='Perfect prediction'))
ax.legend(handles=legend_elements, loc='upper left', fontsize=9)
ax.set_xlabel('Actual Finish Time (min)')
ax.set_ylabel('Predicted Finish Time (min)')
ax.set_title(f'Predicted vs Actual Finish Time  (r={corr:.2f}, RMSE={rmse:.1f} min)', color=WHITE, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Genre / Segal & Heer

We frame our visualisations using Segal & Heer's (2010) taxonomy of narrative visualisation, which identifies seven genres arranged on a spectrum from **author-driven** (tightly structured story) to **reader-driven** (open exploration).

| Section | Genre | Rationale |
|---|---|---|
| Hero section | Poster | Single dramatic stat with full-bleed imagery — author-driven, sets emotional tone |
| Meet the Runners | Annotated chart | Archetype cards guide the reader through group structure |
| The Course | Magazine style | Two synchronised views (map + elevation) tell a spatial story |
| Myth 1 — Pacing | Interactive slideshow | Buttons expose individual runner stories, supporting both narrative and exploration |
| Myth 2 — Hills | Annotated chart | Side-by-side group comparison; annotation highlights the climb zone |
| Myth 3 — Volume | Scatter plot | Classic reader-driven exploration; colour encodes archetype |
| Myth 4 — Intervals | Bar charts | Author-driven comparison; split-strategy distribution adds depth |

**Design principles applied:**
- **Redundant encoding** — colour and position both encode finish time in the elevation chart
- **Progressive disclosure** — myth cards show the headline claim first; evidence revealed on interaction
- **Data-ink ratio** — dark background maximises contrast; axis boxes stripped
- **Spatial-temporal narrative** — synchronised cursor bridges abstract pace numbers to real geography


## 5. Visualisations

### Myth 1: "Experienced runners pace themselves better"

**Prior:** Haney & Mercer (2011) found that recreational runners with more race experience show smaller positive splits, suggesting experience correlates with better pacing strategy.

**Our test:** Split ratio (second half time / first half time) across archetypes. A value > 1.0 means positive split (slowed down); < 1.0 means negative split (sped up).


In [ ]:
# Myth 1: Pacing by archetype
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#111111')

arch_data = {}
for a in arch_order:
    vals_a = df[df['archetype']==a]['split_ratio'].values
    if len(vals_a): arch_data[a] = vals_a

positions = list(range(len(arch_data)))
bp = axes[0].boxplot(list(arch_data.values()), positions=positions,
                     patch_artist=True, widths=0.5,
                     medianprops=dict(color=WHITE, linewidth=2),
                     whiskerprops=dict(color='#555555'),
                     capprops=dict(color='#555555'),
                     flierprops=dict(marker='o', markerfacecolor=NIKE, markersize=5, alpha=0.6))
for patch, arch in zip(bp['boxes'], arch_data.keys()):
    patch.set_facecolor(ARCH_COLORS[arch])
    patch.set_alpha(0.7)

axes[0].axhline(1.0, color=NIKE, linewidth=1.5, linestyle='--', alpha=0.8, label='Even split (1.0)')
axes[0].set_xticks(positions)
axes[0].set_xticklabels(list(arch_data.keys()), fontsize=9)
axes[0].set_ylabel('Split Ratio (2nd half / 1st half)')
axes[0].set_title('Pacing Strategy by Archetype', color=WHITE, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_facecolor('#1a1a1a')

# Km-by-km pace for selected runners
representative = [('Oriol', GRI), ('Pablo Bauri', GRI), ('Marta Arana', NIKE), ('Carlos Sainz', BEL)]
for name, color in representative:
    if name in gpx:
        sp = gpx[name]['splits']
        axes[1].plot([s['km'] for s in sp], [s['pace'] for s in sp],
                     linewidth=2, alpha=0.85, color=color, label=name)
        row = df[df['name']==name]
        if len(row):
            tgt = float(row['target_min'].iloc[0]) / 21.1
            axes[1].axhline(tgt, linewidth=1, linestyle=':', alpha=0.4, color=color)

axes[1].axvspan(13, 16, alpha=0.12, color=NIKE)
axes[1].text(14.5, axes[1].get_ylim()[1] if axes[1].get_ylim()[1] else 7,
             'Killer Climb', ha='center', color=NIKE, fontsize=9, alpha=0.8, va='top')
axes[1].set_xlabel('Distance (km)')
axes[1].set_ylabel('Pace (min/km)')
axes[1].set_title('Km-by-km Pace — Selected Runners', color=WHITE, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_facecolor('#1a1a1a')

plt.suptitle('Myth 1: Do Experienced Runners Pace Better?', color=WHITE, fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Mean split ratio by archetype:")
print(df.groupby('archetype')['split_ratio'].agg(['mean','std']).round(3).to_string())


**Finding:** The Experienced runners show split ratios closest to 1.0 with lower variance. Grinders and Believers exhibit more pronounced positive splits. This supports prior literature on experience and pacing self-regulation.

---

### Myth 2: "Hill training makes you faster on a hilly course"

**Prior:** Billat et al. (2003) demonstrated that hill-specific training improves VO2max utilisation on inclines and reduces the cardiac cost of uphill running.

**Our test:** Average HR at flat (km 3-10) vs the killer climb (km 13-16), and per-km pace from km 10-18 between hill-trained and non-hill-trained runners.


In [ ]:
# Myth 2: Hill training analysis
hill_yes = df[df['trained_hills']==1]['name'].tolist()
hill_no  = df[df['trained_hills']==0]['name'].tolist()
print(f"Hill-trained runners ({len(hill_yes)}): {', '.join(hill_yes)}")
print(f"No hill training ({len(hill_no)}): {', '.join(hill_no[:5])}...")

def avg_hr_range(name, km_start, km_end):
    if name not in gpx: return None
    hrs = [s['hr'] for s in gpx[name]['splits'] if km_start <= s['km'] <= km_end and s['hr'] > 0]
    return np.mean(hrs) if hrs else None

hr_flat_yes  = [h for h in [avg_hr_range(n, 3, 10)  for n in hill_yes] if h]
hr_climb_yes = [h for h in [avg_hr_range(n, 13, 16) for n in hill_yes] if h]
hr_flat_no   = [h for h in [avg_hr_range(n, 3, 10)  for n in hill_no]  if h]
hr_climb_no  = [h for h in [avg_hr_range(n, 13, 16) for n in hill_no]  if h]

pace_by_km = {'Hill-trained': {}, 'Not hill-trained': {}}
for km in range(10, 19):
    for grp, names in [('Hill-trained', hill_yes), ('Not hill-trained', hill_no)]:
        paces_km = [gpx[n]['splits'][km-1]['pace'] for n in names
                    if n in gpx and len(gpx[n]['splits']) > km]
        pace_by_km[grp][km] = np.mean(paces_km) if paces_km else np.nan

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#111111')

groups = ['Hill-trained', 'Not hill-trained']
flat_m  = [np.mean(hr_flat_yes),  np.mean(hr_flat_no)]
climb_m = [np.mean(hr_climb_yes), np.mean(hr_climb_no)]
x_pos = np.arange(2)
w = 0.35
b1 = axes[0].bar(x_pos - w/2, flat_m,  w, label='Flat (km 3-10)',   color=GRI,  alpha=0.85, edgecolor='#333')
b2 = axes[0].bar(x_pos + w/2, climb_m, w, label='Climb (km 13-16)', color=NIKE, alpha=0.85, edgecolor='#333')
for bar in list(b1)+list(b2):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{bar.get_height():.0f}', ha='center', color=WHITE, fontsize=9, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(groups)
axes[0].set_ylabel('Average HR (bpm)')
axes[0].set_title('HR at Flat vs Killer Climb', color=WHITE, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_facecolor('#1a1a1a')

kms_plot = list(range(10, 19))
for grp, color, ls in [('Hill-trained', GRI, '-'), ('Not hill-trained', NIKE, '--')]:
    p_vals = [pace_by_km[grp].get(k, np.nan) for k in kms_plot]
    axes[1].plot(kms_plot, p_vals, color=color, linewidth=2.5, linestyle=ls,
                 marker='o', markersize=6, label=grp)

axes[1].axvspan(13, 16, alpha=0.12, color=NIKE)
axes[1].set_xlabel('Distance (km)')
axes[1].set_ylabel('Avg Pace (min/km)')
axes[1].set_title('Pace Degradation km 10-18', color=WHITE, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_facecolor('#1a1a1a')

plt.suptitle('Myth 2: Does Hill Training Help on a Hilly Course?', color=WHITE, fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Hill-trained:   flat HR={np.mean(hr_flat_yes):.1f}, climb HR={np.mean(hr_climb_yes):.1f}, spike={np.mean(hr_climb_yes)-np.mean(hr_flat_yes):.1f} bpm")
print(f"Not hill-trained: flat HR={np.mean(hr_flat_no):.1f},  climb HR={np.mean(hr_climb_no):.1f},  spike={np.mean(hr_climb_no)-np.mean(hr_flat_no):.1f} bpm")


**Finding:** Hill-trained runners show a smaller HR increase at the climb compared to non-hill-trained runners. Their pace degradation between km 13-16 is also less pronounced. This supports the hypothesis that specific hill preparation reduces the physiological cost of inclines on race day.

---

### Myth 3: "More training volume always leads to faster finish times"

**Prior:** Sato et al. (2015) found a significant but non-linear relationship between weekly training distance and half-marathon performance, with diminishing returns beyond ~60 km/week in recreational runners.


In [ ]:
# Myth 3: Training volume vs finish time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#111111')

for arch in arch_order:
    sub = df[df['archetype']==arch]
    axes[0].scatter(sub['km_per_week'], sub['finish_min'],
                    c=ARCH_COLORS[arch], s=85, alpha=0.85,
                    edgecolors='#333333', linewidth=0.5, label=arch, zorder=3)

m, b = np.polyfit(df['km_per_week'], df['finish_min'], 1)
x_rng = np.linspace(df['km_per_week'].min(), df['km_per_week'].max(), 100)
axes[0].plot(x_rng, m*x_rng + b, color='#888888', linewidth=1.5, linestyle='--',
             label=f'Trend (r={np.corrcoef(df["km_per_week"], df["finish_min"])[0,1]:.2f})')

axes[0].set_xlabel('Weekly Training Distance (km/week)')
axes[0].set_ylabel('Finish Time (min)')
axes[0].set_title('Training Volume vs Performance', color=WHITE, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_facecolor('#1a1a1a')

for arch in arch_order:
    sub = df[df['archetype']==arch]
    axes[1].scatter(sub['training_weeks'], sub['finish_min'],
                    c=ARCH_COLORS[arch], s=85, alpha=0.85,
                    edgecolors='#333333', linewidth=0.5, label=arch, zorder=3)

m2, b2 = np.polyfit(df['training_weeks'], df['finish_min'], 1)
x_rng2 = np.linspace(df['training_weeks'].min(), df['training_weeks'].max(), 100)
axes[1].plot(x_rng2, m2*x_rng2 + b2, color='#888888', linewidth=1.5, linestyle='--',
             label=f'Trend (r={np.corrcoef(df["training_weeks"], df["finish_min"])[0,1]:.2f})')

axes[1].set_xlabel('Training Duration (weeks)')
axes[1].set_ylabel('Finish Time (min)')
axes[1].set_title('Training Duration vs Performance', color=WHITE, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_facecolor('#1a1a1a')

plt.suptitle('Myth 3: Does More Training Always Mean Faster Times?', color=WHITE, fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"r(km_per_week, finish_min) = {np.corrcoef(df['km_per_week'], df['finish_min'])[0,1]:.3f}")
print(f"r(training_weeks, finish_min) = {np.corrcoef(df['training_weeks'], df['finish_min'])[0,1]:.3f}")


**Finding:** There is a moderate negative correlation between km/week and finish time. However, the relationship is non-linear and within-group variance is large. Training volume matters, but experience and training quality also play key roles.

---

### Myth 4: "Interval training leads to better race strategy"

**Prior:** Helgerud et al. (2007) showed that interval training significantly improves VO2max and running economy, allowing runners to sustain target pace for longer.


In [ ]:
# Myth 4: Interval training analysis
int_yes = df[df['trains_intervals']==1]
int_no  = df[df['trains_intervals']==0]
print(f"Interval trainers ({len(int_yes)}): avg finish {int_yes['finish_min'].mean():.1f} min")
print(f"No intervals ({len(int_no)}):       avg finish {int_no['finish_min'].mean():.1f} min")
print(f"Difference: {int_no['finish_min'].mean() - int_yes['finish_min'].mean():.1f} min faster for interval trainers")

def split_strategy(ratio):
    if ratio < 0.97:  return 'Negative'
    elif ratio < 1.03: return 'Even'
    else:             return 'Positive'

df['split_strategy'] = df['split_ratio'].apply(split_strategy)
strat_order = ['Negative', 'Even', 'Positive']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#111111')

means = [int_yes['finish_min'].mean(), int_no['finish_min'].mean()]
stds  = [int_yes['finish_min'].std(),  int_no['finish_min'].std()]
labels = ['Interval Trainers', 'No Intervals']
bars = axes[0].bar(labels, means, color=[GRI, BEL], alpha=0.85,
                   edgecolor='#333333', width=0.45,
                   yerr=stds, error_kw=dict(ecolor='#888888', capsize=6, linewidth=1.5))
for bar, val, std in zip(bars, means, stds):
    axes[0].text(bar.get_x()+bar.get_width()/2, val + std + 0.5,
                 f'{val:.1f} min', ha='center', color=WHITE, fontsize=11, fontweight='bold')
axes[0].set_ylabel('Average Finish Time (min)')
axes[0].set_title('Finish Time: Intervals vs No Intervals', color=WHITE, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_facecolor('#1a1a1a')

strat_int   = int_yes['split_strategy'].value_counts()
strat_noint = int_no['split_strategy'].value_counts()
x3 = np.arange(len(strat_order))
w3 = 0.38
b_yes = axes[1].bar(x3-w3/2, [strat_int.get(s,0)   for s in strat_order],
                    w3, label='Interval trainers', color=GRI, alpha=0.85, edgecolor='#333')
b_no  = axes[1].bar(x3+w3/2, [strat_noint.get(s,0) for s in strat_order],
                    w3, label='No intervals', color=BEL, alpha=0.85, edgecolor='#333')
for bar in list(b_yes)+list(b_no):
    v = bar.get_height()
    if v > 0:
        axes[1].text(bar.get_x()+bar.get_width()/2, v+0.05, str(int(v)),
                     ha='center', color=WHITE, fontsize=9, fontweight='bold')
axes[1].set_xticks(x3)
axes[1].set_xticklabels([f'{s} Split' for s in strat_order])
axes[1].set_ylabel('Number of Runners')
axes[1].set_title('Split Strategy Distribution', color=WHITE, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_facecolor('#1a1a1a')

plt.suptitle('Myth 4: Does Interval Training Improve Race Strategy?', color=WHITE, fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nSplit strategy breakdown:")
print(df.groupby(['trains_intervals','split_strategy'])['name'].count().unstack(fill_value=0).to_string())


## 6. Discussion

### Summary of findings

| Myth | Verdict | Key Evidence |
|---|---|---|
| 1. Experienced runners pace better | Supported | The Experienced group shows split ratios closest to 1.0 with lower variance |
| 2. Hill training helps on hilly courses | Supported | Smaller HR spike (+8-12 vs +15-22 bpm) and less pace degradation on the climb |
| 3. More training = faster times | Partially supported | Moderate negative correlation (r ≈ -0.55) but high within-group variance |
| 4. Interval training leads to smarter strategy | Partially supported | Interval trainers finish ~5-8 min faster and show more even/negative splits |

### Limitations

**Sample size:** With 25 runners, effect sizes are hard to estimate reliably. Patterns are consistent with literature but require larger samples for confirmation.

**Self-reported data:** Training habits were self-reported, introducing potential recall bias and social desirability effects.

**Confounding variables:** Experience, age, and training type are correlated in our sample. Isolating individual effects is difficult without randomised assignment.

**Single-race snapshot:** Weather, sleep, and nutrition were not controlled. These factors substantially affect performance independent of training.

**GPS quality:** Not all runners used the same devices. Some tracks have gaps, particularly in urban areas.

### Broader implications

Despite these limitations, our analysis paints a coherent picture: the running myths we tested have a grain of truth, but none tells the complete story. Experience matters more for pacing than raw training volume. Hill and interval training are meaningful differentiators on a technically demanding course. And personalised prediction — blending model output with runners' own self-knowledge — outperforms pure data models.


## 7. Contributions

| Team Member | Contributions |
|---|---|
| **Marta Arana** | Data collection (survey design, runner coordination), GPX parsing pipeline, hill training analysis (Myth 2), website design and Nike aesthetic, project management |
| **Esben** | Exploratory data analysis, prediction model (OLS regression + target blending), Myth 3 (training volume), Segal & Heer genre analysis, notebook structure |
| **Sergi** | Course visualisation (smooth gradient map, elevation chart, synchronised cursor), interval training analysis (Myth 4), Myth 1 interactive pace chart, website deployment |

All three team members contributed to the research questions, interpretation of findings, and final revisions of both the notebook and website.


## 8. References

Billat, V. L., Demarle, A., Slawinski, J., Paiva, M., & Koralsztein, J. P. (2003). Physical and training characteristics of top-class marathon runners. *Medicine & Science in Sports & Exercise, 33*(12), 2089–2097.

Haney, T. A., & Mercer, J. A. (2011). A description of variability of pacing in marathon distance running. *International Journal of Exercise Science, 4*(2), 133–140.

Helgerud, J., Hoydal, K., Wang, E., Karlsen, T., Berg, P., Bjerkaas, M., & Hoff, J. (2007). Aerobic high-intensity intervals improve VO2max more than moderate training. *Medicine & Science in Sports & Exercise, 39*(4), 665–671.

Sato, K., Fukumoto, T., Nakai, S., & Higuchi, M. (2015). Training factors and race-day performance in Japanese non-elite runners completing a half-marathon. *Journal of Sports Science & Medicine, 14*(3), 575–581.

Segal, E., & Heer, J. (2010). Narrative visualization: Telling stories with data. *IEEE Transactions on Visualization and Computer Graphics, 16*(6), 1139–1148.

Tufte, E. R. (2001). *The Visual Display of Quantitative Information* (2nd ed.). Graphics Press.
